# 3D Video Suitability — reconstruction-grounded ML training


This Colab Free notebook trains only from a versioned JSONL dataset whose targets
come from a documented reconstruction/human-review protocol. It rejects heuristic
scores as targets and rejects synthetic targets by default. The Random Forest is the
first baseline; frozen ResNet-18 embeddings and the temporal LSTM are optional.


Outputs, split manifests, caches, checkpoints, metrics, and hashes are written to
Google Drive. Synthetic smoke-test metrics are never real accuracy.


In [ ]:
# Configuration — edit these paths before running later cells.

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')

PROJECT_DIR = DRIVE_ROOT / '3D-video-suitability-analyzer'

CO3D_ROOT = DRIVE_ROOT / 'co3d_data'

DATASET_JSONL = DRIVE_ROOT / 'video_suitability_ml/dataset_v1.jsonl'

OUTPUT_ROOT = DRIVE_ROOT / 'video_suitability_ml/runs'

RUN_ID = 'genuine_labels_run_001'  # use a new ID for a new experiment

SPLIT_SEED = 20261117

CLASSIFICATION_THRESHOLD = 70.0  # fixed by label protocol; do not tune on test

RUN_RANDOM_FOREST = True

RUN_EMBEDDING_EXTRACTION = False  # enable after the tabular baseline

RUN_TEMPORAL_LSTM = False

ALLOW_SYNTHETIC_SMOKE = False  # only for explicit infrastructure smoke tests

EMBEDDING_BATCH_SIZE = 32

LSTM_BATCH_SIZE = 16

LSTM_MAX_EPOCHS = 30

RUN_DIR = OUTPUT_ROOT / RUN_ID

print({'project': str(PROJECT_DIR), 'dataset': str(DATASET_JSONL), 'run': str(RUN_DIR)})


In [ ]:
# Mount persistent Drive and verify paths before installing anything.

from google.colab import drive

drive.mount('/content/drive')

if not PROJECT_DIR.is_dir():

    raise FileNotFoundError(f'Copy/clone this repository to {PROJECT_DIR}')

if not DATASET_JSONL.is_file():

    raise FileNotFoundError(f'Create/copy the versioned labeled JSONL to {DATASET_JSONL}')

RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Persistent run directory:', RUN_DIR)


In [ ]:
# Install only lightweight missing packages; Colab normally provides torch/torchvision.

import json, os, platform, shutil, subprocess, sys, time

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR), '--no-deps'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.3,<2', 'joblib>=1.3,<2'])

import joblib, numpy as np, sklearn, torch

try:

    import torchvision

    torchvision_status = torchvision.__version__

except Exception as exc:

    torchvision_status = f'unavailable: {exc}'

disk = shutil.disk_usage('/content')

memory_gib = None

try:

    memory_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30

except (ValueError, OSError, AttributeError):

    pass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

runtime = {

    'utc_epoch_seconds': time.time(), 'python': platform.python_version(),

    'torch': torch.__version__, 'torchvision': torchvision_status,

    'scikit_learn': sklearn.__version__, 'device': DEVICE,

    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,

    'memory_gib': memory_gib, 'disk_free_gib': disk.free / 2**30,

}

print(runtime)

(RUN_DIR / 'runtime_environment.json').write_text(json.dumps(runtime, indent=2) + '\n')

if (RUN_EMBEDDING_EXTRACTION or RUN_TEMPORAL_LSTM) and torchvision_status.startswith('unavailable'):

    raise RuntimeError('torchvision is required for frozen ResNet-18 extraction')


In [ ]:
# Validate schema and label provenance. Missing targets stop genuine training.

from video_suitability.ml.schema import read_jsonl

records = read_jsonl(

    DATASET_JSONL, require_targets=True,

    allow_synthetic_targets=ALLOW_SYNTHETIC_SMOKE,

)

scopes = {record.target.provenance.is_synthetic for record in records}

if len(scopes) != 1:

    raise ValueError('Synthetic and genuine targets cannot be mixed')

if scopes == {True} and not ALLOW_SYNTHETIC_SMOKE:

    raise ValueError('Synthetic smoke data is disabled')

print({

    'records': len(records), 'categories': len({r.category for r in records}),

    'split_groups': len({r.split_group_id for r in records}),

    'target_kinds': sorted({r.target.provenance.kind for r in records}),

    'training_scope': 'synthetic_smoke_only' if scopes == {True} else 'genuine_labels',

})


In [ ]:
# Create once, then freeze and validate the physical-object/scene group split.

from video_suitability.ml.splits import SplitManifest, assert_no_group_leakage, grouped_split

split_path = RUN_DIR / 'split_manifest.json'

if split_path.exists():

    split_manifest = SplitManifest.read(split_path)

else:

    split_manifest = grouped_split(records, seed=SPLIT_SEED)

    split_manifest.write(split_path)

assert_no_group_leakage(records, split_manifest)

print({name: len(getattr(split_manifest, f'{name}_sequence_ids')) for name in ['train','validation','test']})


In [ ]:
# First baseline: video-level numerical features, train-only median imputation.

from video_suitability.ml.artifacts import load_and_validate_manifest

from video_suitability.ml.tabular import train_random_forest

rf_root = RUN_DIR / 'random_forest'

rf_artifact = rf_root / 'random_forest_numeric_cv__1.0'

if RUN_RANDOM_FOREST:

    if (rf_artifact / 'artifact_manifest.json').is_file():

        print('Using completed RF artifact:', load_and_validate_manifest(rf_artifact))

    elif rf_artifact.exists():

        raise RuntimeError('Incomplete RF artifact exists; preserve it for diagnosis and choose a new RUN_ID')

    else:

        rf_result = train_random_forest(

            records, split_manifest, rf_root,

            allow_synthetic_targets=ALLOW_SYNTHETIC_SMOKE,

            random_state=SPLIT_SEED,

            classification_threshold=CLASSIFICATION_THRESHOLD,

        )

        print(rf_result)

else:

    print('Random Forest disabled by configuration')


In [ ]:
# Optional frozen ImageNet ResNet-18 embeddings; cache keys include files/config.

# Re-running this cell loads completed sequence caches instead of recomputing them.

from video_suitability.ml.features import EmbeddingCache, ResNet18FrameEmbedder

from video_suitability.ml.schema import write_jsonl

embeddings_by_sequence = {}

if RUN_EMBEDDING_EXTRACTION or RUN_TEMPORAL_LSTM:

    cache = EmbeddingCache(OUTPUT_ROOT / 'embedding_cache_resnet18_v1')

    embedder = ResNet18FrameEmbedder(device=DEVICE, batch_size=EMBEDDING_BATCH_SIZE)

    cache_hits = 0

    for index, record in enumerate(records, 1):

        paths = [Path(p) if Path(p).is_absolute() else CO3D_ROOT / p for p in record.sampled_frame_paths]

        missing = [str(path) for path in paths if not path.is_file()]

        if missing:

            raise FileNotFoundError(f'{record.sequence_id} missing frame: {missing[0]}')

        key, values, hit = embedder.extract_cached(record.sequence_id, paths, cache)

        cache_hits += int(hit)

        embeddings_by_sequence[record.sequence_id] = values

        record.embedding_cache_key = key

        record.embedding_dimension = values.shape[1]

        if index % 25 == 0 or index == len(records):

            print(f'embeddings {index}/{len(records)}; cache hits={cache_hits}')

    write_jsonl(records, RUN_DIR / 'dataset_with_embedding_keys.jsonl', allow_synthetic_targets=ALLOW_SYNTHETIC_SMOKE)

else:

    print('Embedding extraction disabled; no ImageNet weights are downloaded by this notebook')


In [ ]:
# Optional bounded temporal LSTM head. Test metrics are computed only after selection.

from video_suitability.ml.temporal import train_temporal_lstm

lstm_root = RUN_DIR / 'temporal_lstm'

lstm_artifact = lstm_root / 'frozen_resnet18_temporal_lstm__1.0'

if RUN_TEMPORAL_LSTM:

    if not embeddings_by_sequence:

        raise RuntimeError('Enable embedding extraction before LSTM training')

    if (lstm_artifact / 'artifact_manifest.json').is_file():

        print('Using completed LSTM artifact:', load_and_validate_manifest(lstm_artifact))

    elif lstm_artifact.exists():

        raise RuntimeError('Incomplete LSTM artifact/checkpoints preserved; diagnose them and choose a new RUN_ID')

    else:

        lstm_result = train_temporal_lstm(

            records, split_manifest, embeddings_by_sequence, lstm_root,

            allow_synthetic_targets=ALLOW_SYNTHETIC_SMOKE,

            batch_size=LSTM_BATCH_SIZE, max_epochs=LSTM_MAX_EPOCHS,

            random_state=SPLIT_SEED, classification_threshold=CLASSIFICATION_THRESHOLD,

            device=DEVICE,

        )

        print(lstm_result)

else:

    print('Temporal LSTM disabled by configuration')


In [ ]:
# Final inventory. Interpret metrics only according to the recorded training scope.

import json

manifests = sorted(RUN_DIR.rglob('artifact_manifest.json'))

inventory = []

for path in manifests:

    manifest = json.loads(path.read_text())

    inventory.append({

        'path': str(path), 'model_type': manifest['model_type'],

        'model_version': manifest['model_version'],

        'training_scope': manifest['training_scope'],

    })

print(json.dumps({'run_dir': str(RUN_DIR), 'artifacts': inventory}, indent=2))

print('All listed files are already in persistent Google Drive storage.')

print('Sparse registration alone does not establish object completeness; retain human review fields.')
